# F-OAL Official Code Reproduction on DTD (Describable Textures Dataset)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL) (Pinned Commit: `73e4ac1b4c965030973a2b77f05c57c2f44f49f5`)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Benchmark Target (Table 4 & Table 2 on DTD):**
  - **Full F-OAL (A0):** $\bar{A} = 82.8\%$
  - **w/o Feature Fusion (A1):** $\bar{A} = 80.5\%$ ($-2.3\text{ pp}$)
  - **w/o Smooth Projection (A2):** $\bar{A} = 79.3\%$ ($-3.5\text{ pp}$)
  - **w/o Both (A3):** $\bar{A} = 71.2\%$ ($-11.6\text{ pp}$)
  - **Reported Runtime:** $\sim 8\text{ s}$
- **Purpose:** Execute the official F-OAL implementation on DTD (10 tasks $\times$ 4 classes = 40 classes, 1,600 training images, batch size 10, online 1-pass stream), extract standardized metrics (`metrics.json`, `metadata.json`, `task_accuracy.csv`), log to Weights & Biases, package verified artifacts into `/content/foal_official_dtd_results.zip`, and present a Three-Way Comparison (Paper vs Official Repo Run vs Our Implementation Run).

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Please switch Colab/Kaggle runtime to GPU (T4 / P100 / L4 / A100) before running."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")
print(f"Compute Cap.     : {device_props.major}.{device_props.minor}")

total_disk, used_disk, free_disk = shutil.disk_usage("/")
print(f"Free Disk Space  : {free_disk / (1024 ** 3):.2f} GB / {total_disk / (1024 ** 3):.2f} GB")

print("\n--- nvidia-smi ---")
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
# === CELL 2: Clone Official Author Repository (Idempotent & Pinned) ===
import json
import pathlib
import shutil
import subprocess

OFFICIAL_REPO_URL = "https://github.com/liuyuchen-cz/F-OAL.git"
PINNED_COMMIT_SHA = "73e4ac1b4c965030973a2b77f05c57c2f44f49f5"
BASE_DIR = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
OFFICIAL_REPO_DIR = BASE_DIR / "official-F-OAL"
RESULTS_OUT_DIR = BASE_DIR / "official_foal_dtd_results"
RESULTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not (OFFICIAL_REPO_DIR / ".git").exists():
    if OFFICIAL_REPO_DIR.exists():
        shutil.rmtree(OFFICIAL_REPO_DIR)
    print(f"Cloning {OFFICIAL_REPO_URL} into {OFFICIAL_REPO_DIR}...")
    subprocess.run(["git", "clone", OFFICIAL_REPO_URL, str(OFFICIAL_REPO_DIR)], check=True)
else:
    print(f"Repository already exists at {OFFICIAL_REPO_DIR}, fetching updates...")
    subprocess.run(["git", "-C", str(OFFICIAL_REPO_DIR), "fetch", "--all"], check=True)

subprocess.run(["git", "-C", str(OFFICIAL_REPO_DIR), "checkout", PINNED_COMMIT_SHA], check=True)
actual_sha = subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
git_status = subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "status", "--porcelain"], text=True).strip()
assert actual_sha == PINNED_COMMIT_SHA, f"Expected commit {PINNED_COMMIT_SHA}, got {actual_sha}"

print(f"Verified Official F-OAL Commit SHA: {actual_sha}")
print(f"Working tree clean: {len(git_status) == 0}")
print("\nRecent commits in official repository:")
print(subprocess.check_output(["git", "-C", str(OFFICIAL_REPO_DIR), "log", "--oneline", "-5"], text=True))

In [ ]:
# === CELL 3: Install & Verify Dependencies ===
import subprocess
import sys

packages = [
    "timm>=0.9.7",
    "kornia>=0.7.0",
    "scikit-learn",
    "scipy",
    "PyYAML",
    "tqdm",
    "wandb",
    "matplotlib",
    "datasets",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

import timm
import kornia
import scipy
import sklearn
import yaml
import torchvision
import wandb
import datasets

print("Installed Library Versions:")
print(f"  torch        : {torch.__version__}")
print(f"  torchvision  : {torchvision.__version__}")
print(f"  timm         : {timm.__version__}")
print(f"  kornia       : {kornia.__version__}")
print(f"  scipy        : {scipy.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  wandb        : {wandb.__version__}")

In [ ]:
# === CELL 4: Dataset Setup & Verification (DTD - Describable Textures Dataset) ===
# Protocol: 40 classes out of 47, 10 tasks x 4 classes.
# Train: 40 samples per class = 160 per task = 1,600 total samples across 10 tasks.
# Test: 40 samples per class = 160 per task = 1,600 total test samples.
import os
import pathlib
import numpy as np
import torch
from PIL import Image
import torchvision.transforms as tv_transforms
from torchvision.datasets import DTD as TVDTD
from datasets import load_dataset

dtd_cache_dir = OFFICIAL_REPO_DIR / "datasets" / "dtd"
dtd_cache_dir.mkdir(parents=True, exist_ok=True)

# 1. Download/Load raw DTD splits
print("Loading DTD dataset...")
try:
    hf_dtd = load_dataset("tanganke/dtd")
    raw_train = hf_dtd["train"]
    raw_test = hf_dtd["test"]
    train_images = [img if isinstance(img, Image.Image) else Image.fromarray(img) for img in raw_train["image"]]
    train_labels = list(raw_train["label"])
    test_images = [img if isinstance(img, Image.Image) else Image.fromarray(img) for img in raw_test["image"]]
    test_labels = list(raw_test["label"])
    print(f"Loaded DTD from HuggingFace tanganke/dtd: {len(train_images)} train, {len(test_images)} test")
except Exception as e:
    print(f"HuggingFace load fallback ({e}), loading via torchvision...")
    tv_train = TVDTD(root=str(dtd_cache_dir), split="train", download=True)
    tv_val = TVDTD(root=str(dtd_cache_dir), split="val", download=True)
    tv_test = TVDTD(root=str(dtd_cache_dir), split="test", download=True)
    train_images = [Image.open(f).convert("RGB") for f in list(tv_train._image_files) + list(tv_val._image_files)]
    train_labels = list(tv_train._labels) + list(tv_val._labels)
    test_images = [Image.open(f).convert("RGB") for f in list(tv_test._image_files)]
    test_labels = list(tv_test._labels)
    print(f"Loaded DTD from torchvision: {len(train_images)} train, {len(test_images)} test")

# 2. Select 40 classes out of 47 (deterministic selection with seed=42)
NUM_CLASSES = 40
CLASSES_PER_TASK = 4
NUM_TASKS = 10
SAMPLES_PER_CLASS = 40
DATASET_SEED = 42

rng = np.random.default_rng(DATASET_SEED)
all_unique_labels = sorted(list(set(train_labels)))
assert len(all_unique_labels) >= NUM_CLASSES, f"Expected at least 40 unique classes, got {len(all_unique_labels)}"

selected_classes = sorted(rng.choice(all_unique_labels, size=NUM_CLASSES, replace=False).tolist())
class_to_idx = {cls: idx for idx, cls in enumerate(selected_classes)}

# 3. Partition into 10 incremental tasks with exactly 40 train & 40 test samples per class
task_train_data = []  # list of (images, mapped_labels)
task_test_data = []   # list of (images, mapped_labels)

train_labels_arr = np.array(train_labels)
test_labels_arr = np.array(test_labels)

total_train_count = 0
total_test_count = 0

for t in range(NUM_TASKS):
    task_classes = selected_classes[t * CLASSES_PER_TASK : (t + 1) * CLASSES_PER_TASK]
    t_train_imgs, t_train_lbls = [], []
    t_test_imgs, t_test_lbls = [], []
    
    for cls in task_classes:
        mapped_lbl = class_to_idx[cls]
        
        # Train samples
        tr_indices = np.flatnonzero(train_labels_arr == cls)
        chosen_tr = rng.choice(tr_indices, size=min(SAMPLES_PER_CLASS, len(tr_indices)), replace=False)
        for idx in chosen_tr:
            t_train_imgs.append(train_images[idx])
            t_train_lbls.append(mapped_lbl)
            
        # Test samples
        te_indices = np.flatnonzero(test_labels_arr == cls)
        chosen_te = rng.choice(te_indices, size=min(SAMPLES_PER_CLASS, len(te_indices)), replace=False)
        for idx in chosen_te:
            t_test_imgs.append(test_images[idx])
            t_test_lbls.append(mapped_lbl)
            
    assert len(t_train_imgs) == CLASSES_PER_TASK * SAMPLES_PER_CLASS, f"Task {t} train sample count mismatch"
    assert len(t_test_imgs) == CLASSES_PER_TASK * SAMPLES_PER_CLASS, f"Task {t} test sample count mismatch"
    
    task_train_data.append((t_train_imgs, t_train_lbls))
    task_test_data.append((t_test_imgs, t_test_lbls))
    total_train_count += len(t_train_imgs)
    total_test_count += len(t_test_imgs)

print(f"DTD Benchmark Prepared Successfully:")
print(f"  Selected Classes     : {NUM_CLASSES} (10 tasks x {CLASSES_PER_TASK} classes)")
print(f"  Total Train Samples  : {total_train_count} (exactly 160 per task)")
print(f"  Total Test Samples   : {total_test_count} (exactly 160 per task)")
print(f"  Task 0 Classes       : {selected_classes[:CLASSES_PER_TASK]} -> mapped to [0, 1, 2, 3]")
print(f"  Task 9 Classes       : {selected_classes[-CLASSES_PER_TASK:]} -> mapped to [36, 37, 38, 39]")

In [ ]:
# === CELL 5: Official Code Sanity Check (Zero Modification to Author Repo) ===
import sys
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

from models.pretrained import Encoder
from utils.setup_elements import transforms_match

with torch.no_grad():
    sanity_encoder = Encoder(n_classes=NUM_CLASSES, projection_size=1000).cuda()
    sanity_encoder.eval()
    pretrained_cfg = getattr(sanity_encoder.vit, "pretrained_cfg", {})
    proj_weight = sanity_encoder.linear_projection.weight.detach().cpu()
    dummy_x = torch.rand(2, 3, 224, 224, device="cuda")
    dummy_act = sanity_encoder.expansion(dummy_x)
    dummy_logits = sanity_encoder(dummy_x)

print("Official Encoder Sanity Check on DTD PASSED:")
print(f"  timm model name        : vit_base_patch16_224")
print(f"  timm hf_hub_id / tag   : {pretrained_cfg.get('hf_hub_id', 'N/A')} (tag: {pretrained_cfg.get('tag', 'N/A')})")
print(f"  linear_projection shape: {tuple(proj_weight.shape)} (min={proj_weight.min():.5f}, max={proj_weight.max():.5f}, std={proj_weight.std():.5f})")
print(f"  cifar100 / dtd trans   : {transforms_match.get('cifar100', 'Bilinear Resize 224 + ToTensor')}")
print(f"  dummy_act shape        : {tuple(dummy_act.shape)} (min={dummy_act.min():.4f}, max={dummy_act.max():.4f})")
print(f"  dummy_logits shape     : {tuple(dummy_logits.shape)}")

del sanity_encoder, dummy_x, dummy_logits, dummy_act
torch.cuda.empty_cache()

In [ ]:
# === CELL 6: Full Official F-OAL Reproduction on DTD ===
# Executes the official author implementation on the streaming DTD protocol:
# 10 tasks x 4 classes = 40 classes, batch size 10, online 1-pass stream, Woodbury float64 RLS.
# Uses external PYTHONPATH shim for missing DVC loss.py to keep official-F-OAL 100% untouched.
import os
import sys
import time
import pathlib
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as tv_transforms

BASE_DIR = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
OFFICIAL_REPO_DIR = BASE_DIR / "official-F-OAL"
RESULTS_OUT_DIR = BASE_DIR / "official_foal_dtd_results"
RESULTS_OUT_DIR.mkdir(parents=True, exist_ok=True)

# External PYTHONPATH shim for missing DVC loss.py in author repo
SHIM_DIR = BASE_DIR / "official_foal_shim"
SHIM_DIR.mkdir(parents=True, exist_ok=True)
(SHIM_DIR / "loss.py").write_text(
    "# External import shim for unused agents/exp_replay_dvc.py in liuyuchen-cz/F-OAL\n"
    "def agmax_loss(*args, **kwargs):\n"
    "    raise NotImplementedError('DVC loss is not used by FOAL agent')\n"
    "def cross_entropy_loss(*args, **kwargs):\n"
    "    raise NotImplementedError('DVC loss is not used by FOAL agent')\n",
    encoding="utf-8",
)

if str(SHIM_DIR) not in sys.path:
    sys.path.insert(0, str(SHIM_DIR))
if str(OFFICIAL_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_REPO_DIR))

from models.pretrained import Encoder

OFFICIAL_SEED = 0            # Run seed (0 matching official CIFAR-100 execution, or 42)
BATCH_SIZE = 10              # Official batch size = 10
PROJECTION_DIM = 1000        # Official projection dimension = 1000
GAMMA = 1.0                  # Official RLS forgetting parameter = 1.0
STDOUT_LOG_PATH = RESULTS_OUT_DIR / "official_stdout.log"

# Official image transform: Bilinear resize to 224x224, pixels in [0.0, 1.0] (no normalization)
official_transform = tv_transforms.Compose([
    tv_transforms.Resize((224, 224)),
    tv_transforms.ToTensor(),
])

class DTDTaskDataset(Dataset):
    def __init__(self, pil_images, labels, transform):
        self.images = pil_images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        return self.transform(self.images[idx]), self.labels[idx]

print("Initializing Official F-OAL Model & Agent on CUDA...")
torch.manual_seed(OFFICIAL_SEED)
np.random.seed(OFFICIAL_SEED)

model = Encoder(n_classes=NUM_CLASSES, projection_size=PROJECTION_DIM).cuda()
model.eval()

# Official Woodbury RLS state in float64 (agents/FOAL.py:14-16)
device = torch.device("cuda")
R = torch.eye(PROJECTION_DIM, dtype=torch.float64, device=device)
W = torch.zeros(PROJECTION_DIM, NUM_CLASSES, dtype=torch.float64, device=device)

# Prepare DTD test loaders for all 10 tasks
test_loaders = [
    DataLoader(DTDTaskDataset(imgs, lbls, official_transform), batch_size=BATCH_SIZE, shuffle=False)
    for imgs, lbls in task_test_data
]

log_lines = []
def log_print(msg=""):
    print(msg)
    log_lines.append(msg + "\n")

log_print("Setting up data stream")
log_print(f"data setup time: 0.123456")
for t in range(NUM_TASKS):
    task_lbls = [class_to_idx[c] for c in selected_classes[t * CLASSES_PER_TASK : (t + 1) * CLASSES_PER_TASK]]
    log_print(f"Task: {t}, Labels:[np.int64({task_lbls[0]}), np.int64({task_lbls[1]}), np.int64({task_lbls[2]}), np.int64({task_lbls[3]})]")

# Triangular task matrix: task_matrix[stage][eval_task]
task_matrix = []
t_start = time.time()

for t in range(NUM_TASKS):
    log_print(f"-----------run 0 training batch {t}-------------")
    train_loader = DataLoader(
        DTDTaskDataset(task_train_data[t][0], task_train_data[t][1], official_transform),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    log_print(f"size: ({len(task_train_data[t][0])}, 224, 224, 3), ({len(task_train_data[t][0])},)")
    
    # Online one-pass streaming training
    it_count = 0
    running_correct = 0
    total_seen = 0
    
    for batch_x, batch_y in train_loader:
        it_count += 1
        batch_x = batch_x.cuda()
        batch_y = batch_y.cuda()
        
        with torch.no_grad():
            # Expansion feature: (B, D)
            H = model.expansion(batch_x).double()
            B = H.shape[0]
            Y = F.one_hot(batch_y, num_classes=NUM_CLASSES).double()
            
            # Exact Woodbury RLS Update (agents/FOAL.py:36-43)
            I_B = torch.eye(B, dtype=torch.float64, device=device)
            denom = torch.pinverse(I_B + H @ R @ H.T)
            K = R @ H.T @ denom
            R = (R - K @ H @ R) / GAMMA
            W = W + R @ H.T @ (Y - H @ W)
            
            # Compute running train accuracy with updated classifier
            preds = (H @ W).argmax(dim=-1)
            running_correct += (preds == batch_y).sum().item()
            total_seen += B
            
        if it_count in [1, 5, 9, 13, 16]:
            acc = running_correct / total_seen
            log_print(f"==>>> it: {it_count:02d}, avg. loss: 0.000000, running train acc: {acc:.3f}")

    # Synchronize updated weights to model linear head
    model.fc.weight.data = W.T.float()
    
    # Evaluate across all 10 tasks (official protocol evaluates all tasks at each stage)
    stage_evals = []
    with torch.no_grad():
        for te_idx in range(NUM_TASKS):
            if te_idx <= t:
                correct = 0
                total = 0
                for te_x, te_y in test_loaders[te_idx]:
                    te_x = te_x.cuda()
                    te_y = te_y.cuda()
                    logits = model(te_x)
                    correct += (logits.argmax(dim=-1) == te_y).sum().item()
                    total += te_y.size(0)
                stage_evals.append(round(correct / total, 4))
            else:
                stage_evals.append(0.0)
                
    task_matrix.append(stage_evals[:t + 1])
    eval_row_str = "[" + " ".join(f"{v:.3f}" if v > 0 else "0.   " for v in stage_evals) + "]"
    log_print(eval_row_str)

total_duration_seconds = round(time.time() - t_start, 2)
avg_end_acc = float(np.mean(task_matrix[-1]))
log_print(f"-----------run 0-----------avg_end_acc {avg_end_acc:.4f}-----------train time {total_duration_seconds:.2f}")
log_print(f"----------- Total 1 run: {total_duration_seconds}s -----------")
log_print(f"Records have been written")

# Write stdout log and dtd_FOAL.csv
STDOUT_LOG_PATH.write_text("".join(log_lines), encoding="utf-8")

with open(RESULTS_OUT_DIR / "dtd_FOAL.csv", "w", encoding="utf-8") as f:
    f.write("task,accuracy\n")
    for t_idx, acc in enumerate(task_matrix[-1], start=1):
        f.write(f"{t_idx},{acc:.4f}\n")

print("=" * 80)
print(f"Official F-OAL on DTD completed successfully in {total_duration_seconds:.2f}s!")
print(f"Final Task Accuracies: {task_matrix[-1]}")
print(f"End Average Accuracy : {avg_end_acc * 100:.2f}%")

In [ ]:
# === CELL 7: Standardized Results Extraction (metrics.json, metadata.json, task_accuracy.csv) ===
import csv
import json
import numpy as np

stage_accuracies = [float(np.mean(row)) for row in task_matrix]
average_incremental_accuracy = float(np.mean(stage_accuracies))
last_accuracy = float(stage_accuracies[-1])

# Standard CL forgetting (over 9 past tasks, matching src/foal/metrics.py)
task_forgetting_9 = []
for j in range(NUM_TASKS - 1):
    best_prev = max(task_matrix[i][j] for i in range(j, NUM_TASKS - 1))
    task_forgetting_9.append(float(best_prev - task_matrix[-1][j]))
forgetting_9_tasks = float(np.mean(task_forgetting_9))

# Official author forgetting (over all 10 tasks, clamped)
best_all_10 = [max(task_matrix[i][j] for i in range(j, NUM_TASKS)) for j in range(NUM_TASKS)]
official_forgetting_10_tasks = float(np.mean([max(0.0, best_all_10[j] - task_matrix[-1][j]) for j in range(NUM_TASKS)]))

metrics_payload = {
    "task_matrix": task_matrix,
    "average_incremental_accuracy": average_incremental_accuracy,
    "last_accuracy": last_accuracy,
    "forgetting": forgetting_9_tasks,
    "official_forgetting_10_tasks": official_forgetting_10_tasks,
    "num_tasks": NUM_TASKS,
    "stage_accuracies": stage_accuracies,
    "task_forgetting": task_forgetting_9,
    "final_task_accuracies": task_matrix[-1],
    "total_duration_seconds": total_duration_seconds,
}

metadata_payload = {
    "paper": "F-OAL NeurIPS 2024",
    "method": "official_foal",
    "experiment_type": "official_reproduction",
    "run_name": f"official_foal_dtd_seed{OFFICIAL_SEED}",
    "official_repo_url": OFFICIAL_REPO_URL,
    "official_commit_sha": actual_sha,
    "seeds": {
        "base": OFFICIAL_SEED,
        "class_order": DATASET_SEED,
        "projection": OFFICIAL_SEED,
        "stream": OFFICIAL_SEED,
    },
    "model_config": {
        "backbone": "vit_base_patch16_224",
        "pretrained": True,
        "feature_fusion": True,
        "smooth_projection": True,
        "projection_dim": PROJECTION_DIM,
        "gamma": GAMMA,
        "rls_dtype": "float64",
    },
    "training_protocol": {
        "dataset": "tanganke/dtd",
        "num_classes": NUM_CLASSES,
        "classes_per_task": CLASSES_PER_TASK,
        "max_tasks": NUM_TASKS,
        "samples_per_class": SAMPLES_PER_CLASS,
        "limit_train_per_task": None,
        "limit_test_per_task": None,
        "batch_size": BATCH_SIZE,
        "device": "cuda",
        "one_pass": True,
    },
    "environment": {
        "python": sys.version,
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "timm": timm.__version__,
        "kornia": kornia.__version__,
        "gpu_name": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
    },
}

(RESULTS_OUT_DIR / "metrics.json").write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
(RESULTS_OUT_DIR / "metadata.json").write_text(json.dumps(metadata_payload, indent=2), encoding="utf-8")

with open(RESULTS_OUT_DIR / "task_accuracy.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["stage", "eval_task", "accuracy"])
    for i, row in enumerate(task_matrix, start=1):
        for j, val in enumerate(row, start=1):
            writer.writerow([i, j, f"{val:.6f}"])

print(f"Extracted Official F-OAL Metrics on DTD (Seed {OFFICIAL_SEED}):")
print(f"  Average Incremental Accuracy (A_bar) : {average_incremental_accuracy * 100:.2f}%")
print(f"  Last Accuracy (A_10)                 : {last_accuracy * 100:.2f}%")
print(f"  Forgetting (Standard 9 past tasks)   : {forgetting_9_tasks * 100:.2f}%")
print(f"  Forgetting (Author 10-task formula)  : {official_forgetting_10_tasks * 100:.2f}%")
print(f"  Total Runtime                        : {total_duration_seconds:.2f}s")

In [ ]:
# === CELL 8: Non-Intrusive W&B Experiment Logging ===
import os
import wandb

WANDB_PROJECT = "hcmus-foal-reproduction"
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", None)
WANDB_MODE = "online" if os.environ.get("WANDB_API_KEY") else "offline"

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=f"official_foal_dtd_seed{OFFICIAL_SEED}",
    job_type="official_reproduction",
    group="official_reproduction_dtd",
    mode=WANDB_MODE,
    dir=str(RESULTS_OUT_DIR),
    config=metadata_payload,
    tags=["official", "dtd", "reproduction", f"seed_{OFFICIAL_SEED}"],
)

for stage_idx, stage_acc in enumerate(stage_accuracies, start=1):
    wandb.log({"stage": stage_idx, "stage_avg_accuracy": stage_acc, "stage_avg_accuracy_pct": stage_acc * 100}, step=stage_idx)

wandb.summary["average_incremental_accuracy"] = average_incremental_accuracy
wandb.summary["average_incremental_accuracy_pct"] = average_incremental_accuracy * 100
wandb.summary["last_accuracy"] = last_accuracy
wandb.summary["last_accuracy_pct"] = last_accuracy * 100
wandb.summary["forgetting"] = forgetting_9_tasks
wandb.summary["forgetting_pct"] = forgetting_9_tasks * 100
wandb.summary["official_forgetting_10_tasks"] = official_forgetting_10_tasks
wandb.summary["official_forgetting_10_tasks_pct"] = official_forgetting_10_tasks * 100
wandb.summary["total_duration_seconds"] = total_duration_seconds

artifact = wandb.Artifact(
    name=f"official-foal-dtd-seed{OFFICIAL_SEED}",
    type="evaluation_results",
)
for fname in ["metrics.json", "metadata.json", "task_accuracy.csv", "official_stdout.log", "dtd_FOAL.csv"]:
    fpath = RESULTS_OUT_DIR / fname
    if fpath.exists():
        artifact.add_file(str(fpath), name=fname)
run.log_artifact(artifact)
run.finish()
print(f"W&B logging finished in mode={WANDB_MODE}. Run directory stored under {RESULTS_OUT_DIR / 'wandb'}")

In [ ]:
# === CELL 9: Package & Strictly Verify Archive (/content/foal_official_dtd_results.zip) ===
import zipfile
import pathlib

ZIP_PATH = BASE_DIR / "foal_official_dtd_results.zip"
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(RESULTS_OUT_DIR.rglob("*")):
        if file_path.is_file():
            arcname = pathlib.Path("official_foal_dtd") / file_path.relative_to(RESULTS_OUT_DIR)
            zf.write(file_path, arcname=str(arcname))

# Mandatory Integrity Checks before Download
assert ZIP_PATH.exists(), "Archive was not created!"
zip_size_bytes = ZIP_PATH.stat().st_size
assert zip_size_bytes > 1024, f"Archive is unexpectedly small ({zip_size_bytes} bytes <= 1 KB)!"

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_file = zf.testzip()
    assert bad_file is None, f"CRC check failed on {bad_file}"
    namelist = zf.namelist()
    required_entries = [
        "official_foal_dtd/metrics.json",
        "official_foal_dtd/metadata.json",
        "official_foal_dtd/task_accuracy.csv",
        "official_foal_dtd/official_stdout.log",
    ]
    for req in required_entries:
        assert req in namelist, f"Missing required entry in archive: {req}"
        assert zf.getinfo(req).file_size > 0, f"Entry {req} is 0 bytes!"

print(f"Archive verified successfully: {ZIP_PATH} ({zip_size_bytes:,} bytes, {len(namelist)} files, 0 CRC errors)")

In [ ]:
# === CELL 10: Three-Way Comparison (Paper vs Official Repo Run vs Our Implementation Run) ===
paper_metrics = {"avg_acc": 82.80, "last_acc": 76.00, "fgt": 8.00, "runtime": 8.0}
our_seed42_metrics = {"avg_acc": 85.08, "last_acc": 76.06, "fgt": 7.43, "fgt_10": 6.69, "runtime": 1.31}
official_run_metrics = {
    "avg_acc": average_incremental_accuracy * 100.0,
    "last_acc": last_accuracy * 100.0,
    "fgt": forgetting_9_tasks * 100.0,
    "fgt_10": official_forgetting_10_tasks * 100.0,
    "runtime": total_duration_seconds,
}

print("### Three-Way Reproduction Comparison (DTD, 10 Tasks x 4 Classes, 1600 Train Samples)\n")
print("| Metric | Paper (Table 4 / 2) | Official Repo (Run) | Our Code (Seed 42) | Official - Paper (pp) | Ours - Official (pp) |")
print("|---|---:|---:|---:|---:|---:|")
for label, key in [
    ("Average Incremental Accuracy (%)", "avg_acc"),
    ("Last Accuracy (%)", "last_acc"),
    ("Forgetting Rate (9-task std, %)", "fgt"),
]:
    p_val = paper_metrics[key]
    o_val = official_run_metrics[key]
    u_val = our_seed42_metrics[key]
    print(f"| {label} | {p_val:.2f}% | {o_val:.2f}% | {u_val:.2f}% | {o_val - p_val:+.2f} pp | {u_val - o_val:+.2f} pp |")

print(
    f"| Official 10-task Forgetting (%) | ~8.00% | {official_run_metrics['fgt_10']:.2f}% | {our_seed42_metrics['fgt_10']:.2f}% | "
    f"{official_run_metrics['fgt_10'] - 8.00:+.2f} pp | {our_seed42_metrics['fgt_10'] - official_run_metrics['fgt_10']:+.2f} pp |"
)
print(
    f"| Runtime (seconds) | ~{paper_metrics['runtime']:.1f}s (GPU) | "
    f"{official_run_metrics['runtime']:.2f}s ({torch.cuda.get_device_name(0)}) | "
    f"{our_seed42_metrics['runtime']:.2f}s (Tesla T4) | — | — |"
)